# Notebook 0 — Flight pre-check

Run this **once** before Notebooks 1–3. It verifies the environment and **downloads the workshop image subset** so later notebooks can stay short and focused on the science.

| Check | Why it matters |
|-------|----------------|
| Python packages | Torch / transformers / Hydra must import |
| API baseline | Workshop env usually already has Vector proxy (`OPENAI_API_KEY` + base URL); optional `.env` overrides only |
| Vector proxy ping | Confirms network + key against `proxy.vectorinstitute.ai` |
| GPU / CUDA | Picks `cpu` vs `gpu_l4` vs `gpu_l4x2` |
| **Sample images** | Mapillary toy set (fetched here with a progress bar if missing) |
| Model IDs | What this hardware profile will load (cached vs download-on-first-use) |

**Does not** download Klein or run diffusion — that stays in Notebook 1.

Next: [Notebook 0.5](00.5_method_comparison.ipynb) (optional method bake-off) → [Notebook 1](01_sample_data_generation.ipynb).

---
## 0. Setup

From the **repo root** (once per machine):

```bash
uv sync --dev --group edge-case-image-generation

# Optional — only if you want your own API keys / base URLs, or HF_TOKEN.
# The workshop machine usually already exports the Vector proxy baseline;
# you do not paste a Vector key into `.env` for the default path.
cp implementations/edge_case_image_generation/.env.example \
   implementations/edge_case_image_generation/.env
```

Select the project kernel, then run the cells below. Sample images are pulled in **§2** (no separate extract script required). The CLI twin still exists if you prefer: `uv run python scripts/extract_mapillary_toy.py`.

In [1]:
from aieng.syn_data.image.bootstrap import bootstrap_project_root
from aieng.syn_data.image.config import load_env


PROJECT_ROOT = bootstrap_project_root()
env_path = load_env(PROJECT_ROOT)
print("PROJECT_ROOT =", PROJECT_ROOT)
print(".env         =", env_path or "(none — OK if workshop env already has API keys)")

/home/coder/synthetic-data-bootcamp/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PROJECT_ROOT = /home/coder/synthetic-data-bootcamp/implementations/edge_case_image_generation
.env         = /home/coder/synthetic-data-bootcamp/implementations/edge_case_image_generation/.env


---
## 1. Knobs

- `DATASET` — Hydra dataset package under `configs/datasets/`.
- `PING_PROXY` — set `False` if you are offline and only want local checks.
- `HARDWARE_OVERRIDE` — leave `None` to auto-recommend from GPUs; or force `"cpu"` / `"gpu_l4"` / `"gpu_l4x2"`.
- `FORCE_REEXTRACT` — wipe and rebuild the toy image subset.
- `MIN_SAMPLES` — treat the cache as incomplete below this count.

### Make it yours — where the configs live

| Want to change… | Edit |
|-----------------|------|
| **Workshop size** (this download + NB2/NB3 counts) | `configs/config.yaml` → `scale`. `1.0` = full run (~1110 images, ~17 min download, NB2 ~2.5 h); `0.2` = bootcamp small mode (~220 images, about a fifth of the time). Set it **before** the data cell below. |
| **Your own dataset** | Copy `configs/datasets/_template/` → `configs/datasets/<your_id>/`, then set `DATASET = "<your_id>"` here and in NB0.5–NB3. |
| **Your own local images** | `kind: local` in `<your_id>/data.yaml`, then drop photos into `data/<your_id>/samples/`: `scene_*.jpg` = clean backgrounds (edit seeds), `<anomaly_id>_*.jpg` = real positives. Optional `labels.json` next to them: `{"traffic_cone_001.jpg": [{"label": "traffic_cone", "bbox_xyxy": [x1, y1, x2, y2]}]}` — NB3 needs these boxes on real positives to score the detector. |
| **Other image sources** | `kind: urls \| voc_zip \| hf_rows \| hf_detection` — field examples in `_template/data.yaml` and the `rdd2022` / `nordland_hf` packages. |
| **Models per GPU** | `configs/hardware/<profile>.yaml` (editor, depth/seg, detector) on top of `configs/default/*.yaml`. |

Anomalies, judge/API and detector pointers sit next to the cells that use them in NB1–NB3.


In [2]:
DATASET = "mapillary_vistas"
PING_PROXY = True
HARDWARE_OVERRIDE = None  # e.g. "gpu_l4" to force
FORCE_REEXTRACT = False
MIN_SAMPLES = 1

---
## 2. Workshop data (Mapillary toy subset)

Checks `data/.../samples/`. If empty (or `FORCE_REEXTRACT`), extracts a small tagged subset via authenticated zip **Range GETs** — not the full ~29 GB archive.
If the folder has fewer `scene_*` photos than `extract_max_generic` in `configs/datasets/mapillary_vistas/data.yaml` (NB2 draws its seed pools from these), only the missing scenes are downloaded.

**How many images:** `extract_max_per_class` (per rare class) + `extract_max_generic` (scenes) in `configs/datasets/mapillary_vistas/data.yaml`, multiplied by `scale` from `configs/config.yaml`. Lowering `scale` after a full download is fine — NB2/NB3 just use fewer of the images already on disk.

**Hugging Face (one-time, before first extract):**
1. Log into [Hugging Face](https://huggingface.co/login).
2. Open the [Mapillary Vistas HF mirror](https://huggingface.co/datasets/candylion/mapillary-vistas-v2) and **accept / approve access** (gated dataset).
3. Auth here with `.env` `HF_TOKEN`, or let the next cell prompt an interactive `huggingface-cli` login.

You do **not** need to download the full archive by hand — this notebook (or the optional CLI) pulls only the toy subset.

In [3]:
import os

from aieng.syn_data.image.preflight import ensure_workshop_data
from huggingface_hub import get_token, login


if not (get_token() or os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN")):
    print("No HF token in env/cache — starting interactive login…")
    login()

sample_paths = ensure_workshop_data(
    PROJECT_ROOT,
    dataset_name=DATASET,
    clean=FORCE_REEXTRACT,
    min_images=MIN_SAMPLES,
)
print(f"Ready: {len(sample_paths)} sample image(s)")

Samples OK: 430 images (320 scene_ seeds, 110 tagged) in /home/coder/synthetic-data-bootcamp/implementations/edge_case_image_generation/data/mapillary_vistas/samples
Ready: 430 sample image(s)


---
## 3. Run preflight

Rows marked **XX** must be fixed before Notebook 1. **!!** warnings are usually OK for a smoke test but will slow you down or block gated downloads.

In [4]:
from aieng.syn_data.image.preflight import run_preflight


report = run_preflight(
    PROJECT_ROOT,
    dataset_name=DATASET,
    ping_proxy=PING_PROXY,
    hardware_override=HARDWARE_OVERRIDE,
)
report.print_table()

HARDWARE = report.recommended_hardware
print(f"\nCopy into later notebooks:  HARDWARE = {HARDWARE!r}  DATASET = {DATASET!r}")

STATUS  CHECK               DETAIL
------  ------------------  ----------------------------------------
OK      python packages     torch 2.14.0+cu130, transformers/openai/hydra OK
OK      env file            /home/coder/synthetic-data-bootcamp/implementations/edge_case_image_generation/.env
OK      Vector API key      vp_7…868d (len=76)
OK      Hugging Face login  token present (needed for gated Mapillary + some model weights)
OK      disk space          54.5 GB free
OK      GPU / CUDA          2× GPU: NVIDIA L4 (22 GB), NVIDIA L4 (22 GB) → recommend gpu_l4x2
OK      sample images       430 images in samples/ (320 scene_ seeds, 110 tagged)
OK      models (config)     profile=gpu_l4x2; depth: depth-anything/Depth-Anything-V2-Large-hf [cached]; seg: nvidia/segformer-b3-finetuned-ade-512-512 [cached]; instruct: black-forest-labs/FLUX.2-klein-4B [cached]; judge: gemini-3-flash-preview (API — no local weights); embed: openai/clip-vit-base-patch32 [cached]
OK      YOLO-World weight   found 

---
## 4. What each later notebook expects

| Notebook | Needs from this preflight |
|----------|---------------------------|
| **0.5** Method comparison | Same key + hardware; optional Nano Banana uses the **same** Vector proxy |
| **1** Single-image loop | Samples + GPU (or patience on CPU) + judge API |
| **2** Batch synth | Dual-GPU helps; CLI: `scripts/run_nb2_batch.py` |
| **3** Detector train/eval | Needs NB2 export under `outputs/<dataset>/nb2/` |

If preflight is **READY**, jump to [Notebook 1](01_sample_data_generation.ipynb) (or [0.5](00.5_method_comparison.ipynb) if you want to compare edit methods first).

In [5]:
assert report.ready, (
    "Preflight failed — fix the XX rows above before continuing. Re-run after editing .env or re-running the data cell."
)
print("All required checks passed. You're clear for Notebook 1.")

All required checks passed. You're clear for Notebook 1.
